In [0]:
%run "/Workspace/Users/soumya.saha221@gmail.com/traffic_data_project/4. Common config"

In [0]:
dbutils.widgets.text(name='env',defaultValue='',label='Enter the Environment')
env = dbutils.widgets.get('env')

####Reading silver raw traffic table

In [0]:
def read_silver_traffic(environment):
    print('Reading raw traffic silver table: ',end='')
    df = spark.readStream.table(f"{environment}catalog.silver.silver_traffic")
    print('Success!')
    return df

####Reading silver raw roads table

In [0]:
def read_silver_roads(environment):
    print('Reading raw roads silver table: ',end='')
    df = spark.readStream.table(f"{environment}catalog.silver.silver_roads")
    print('Success!')
    return df

####Creating motor vehicle intensity

In [0]:
def mv_intensity(df):
    print('MV intensity calculation: ',end='')
    from pyspark.sql.functions import col
    df_intensity = df.withColumn('MV_intensity', col('MV_total_count')/col('Link_length_km'))
    print('Success!')
    return df_intensity


####Creating load time column

In [0]:
def gold_load_time(df):
    print('Gold loading time column added: ', end='')
    from pyspark.sql.functions import current_timestamp
    df_time  = df.withColumn('gold_load_time', current_timestamp())
    print('Success!')
    return df_time

####Creating gold aggs

In [0]:
def gold_agg_traffic(df):
    print('Gold aggregation: ',end='')
    from pyspark.sql.functions import sum, avg
    df_agg = df.groupBy('Region_name').agg(sum('EV_total_count').alias('EV_total_count_per_region')
                                           ,sum('MV_total_count').alias('MV_total_count_per_region'))
    print('Success!')
    return df_agg

def gold_agg_roads(df):
    print('Gold aggregation: ',end='')
    from pyspark.sql.functions import sum, avg
    df_agg = df.groupBy('Region_name').agg((sum('All_motor_vehicles')/sum('Total_link_length_km')).alias('MV_intensity_per_region'))
    print('Success!')
    return df_agg


####Writing to gold tables

In [0]:
def write_to_goldtraffic(df, environment):
    print('Writing to gold table: ', end='')
    df.writeStream.format('delta')\
    .outputMode('complete')\
    .option('checkpointLocation', f"{checkpoint}/goldTrafficLoad/checkpnt")\
      .trigger(availableNow = True) \
    .toTable(f"{environment}catalog.gold.gold_traffic").awaitTermination()
    print('Success!!!')

def write_to_goldroads(df, environment):
    print('Writing to gold table: ', end='')
    df.writeStream.format('delta')\
    .outputMode('complete')\
    .option('checkpointLocation', f"{checkpoint}/goldRoadsLoad/checkpnt")\
      .trigger(availableNow = True) \
    .toTable(f"{environment}catalog.gold.gold_roads").awaitTermination()
    print('Success!!!')    

####Calling all functions

In [0]:
#Loading gold traffic table
dftraffic = read_silver_traffic(env)
dfintensity = mv_intensity(dftraffic)
dftraffictime = gold_load_time(dfintensity)
dftraffictimeagg = gold_agg_traffic(dftraffictime)
write_to_goldtraffic(dftraffictimeagg, env)

#Loading gold roads table
dfroads = read_silver_roads(env)
dfroadstime = gold_load_time(dfroads)
dfroadstimeagg = gold_agg_roads(dfroadstime)
write_to_goldroads(dfroadstimeagg, env)